# wrangle / portfolio / profile

**Intent**
- Takes `portfolio_profile_risky_share_pct`, in percent.
- Takes `portfolio_profile_withdrawal_rate_pct`, in percent of initial wealth, constant in real terms.
- Outputs `portfolio_profile_return_pct` and `portfolio_profile_volatility_pct`, in percent a year.
- Outputs `portfolio_profile_shortfall_probability_pct`, the chance of running out within 30 years.
- The shortfall is simulated from that same return and volatility, so they cannot disagree.
- Lets any mix and withdrawal be compared on risk, return and shortfall.
- Acceptance holds return and volatility to 0.05 points, and shortfall to 10 points.

**Data**
- Personal data is git-ignored in `data/`, with synthetic placeholders committed.
- `training` is thirty share and withdrawal combinations, `validation` is twenty further, and `backtest` is twenty across the range.
- `backtest` is held back for acceptance.
- `golden` is shares from 0 to 100 percent with withdrawal rates from 2 to 6, then a request with an input missing.

**Status**
- Two assets and their correlation are recovered from synthetic points.
- The mix return is a line and its variance a quadratic in the risky share.
- Shortfall is a logistic curve fitted to a fixed seed simulation, and the returns are real.

**Future**
- Overall: the work still to do before the first release:
  - Add `query_postgres` to `workbench.data` and load the real series in place of the synthetic ones.
  - Draw returns by stationary block bootstrap from the `portfolio/return` history, re-centred on its forward base.
  - Draw inflation from `household/costs` jointly with returns, and use survival from `portfolio/withdrawal` instead of 30 years.
  - Report the shortfall probability, the expected shortfall and the fifth percentile terminal wealth.
  - Try a spending cut rule, since being able to cut helps ruin more than any asset choice.
  - End in one table per candidate risky share, with sensitivities and the worst drawdown.
  - Decide first the spending, horizon, tolerance for a fall, other income and bequest, which the model cannot.
  - Fix the limits against real data, then register and review before any release.

## Setup ([6.7.1](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#671-setup))

In [ ]:
from os.path import join

from workbench import setup_workspace

ws = setup_workspace("wrangle/portfolio/profile", {"kind": "served"})


GOLDEN = "portfolio_profile_golden"

## Data ([6.7.2](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#672-data))

In [ ]:
from math import sqrt

import numpy as np

from workbench import data_load

YEARS = 30
PATHS = 2000


def mix(share):
    weight = share / 100.0
    variance = (weight * 15.0) ** 2 + ((1.0 - weight) * 5.0) ** 2 + 2.0 * 0.2 * weight * (1.0 - weight) * 15.0 * 5.0
    return 2.0 + 4.0 * weight, sqrt(variance)


def shortfall(share, rate):
    mean, volatility = mix(share)
    returns = np.random.default_rng(0).normal(mean / 100.0, volatility / 100.0, (PATHS, YEARS))
    wealth = np.full(PATHS, 100.0)
    for year in range(YEARS):
        wealth = (wealth - rate) * (1.0 + returns[:, year])
        wealth = np.where(wealth > 0.0, wealth, 0.0)
    return float(np.mean(wealth <= 0.0) * 100.0)


def labelled(shares, rates):
    pairs = [(share, rate) for share in shares for rate in rates]
    return {
        ws.inputs[0]: [share for share, _ in pairs],
        ws.inputs[1]: [rate for _, rate in pairs],
        ws.outputs[0]: [round(mix(share)[0], 6) for share, _ in pairs],
        ws.outputs[1]: [round(mix(share)[1], 6) for share, _ in pairs],
        ws.outputs[2]: [round(shortfall(share, rate), 4) for share, rate in pairs],
    }


training_pl, training_path = data_load(ws, "training", "portfolio_profile_training", run=1, source="builder", build=lambda: labelled([10.0, 30.0, 50.0, 70.0, 90.0], [2.0, 3.0, 4.0, 5.0, 6.0, 7.0]))

validation_pl, validation_path = data_load(ws, "validation", "portfolio_profile_validation", run=1, source="builder", build=lambda: labelled([20.0, 40.0, 60.0, 80.0], [2.5, 3.5, 4.5, 5.5, 6.5]))

backtest_pl, backtest_path = data_load(ws, "backtest", "portfolio_profile_backtest", run=1, source="builder", build=lambda: labelled([15.0, 35.0, 55.0, 75.0, 95.0], [3.25, 4.25, 5.25, 6.25]))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Train ([6.7.4](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#674-train))

In [ ]:
from scipy.optimize import curve_fit

weights = training_pl[ws.inputs[0]].to_numpy() / 100.0
rates = training_pl[ws.inputs[1]].to_numpy()
means = training_pl[ws.outputs[0]].to_numpy()
volatilities = training_pl[ws.outputs[1]].to_numpy()
return_slope, return_intercept = np.polyfit(weights, means, 1)
variance_square, variance_linear, variance_constant = np.polyfit(weights, volatilities**2, 2)
fitted = curve_fit(
    lambda data, slope, m0, m1, m2: 100.0 / (1.0 + np.exp(np.minimum(-slope * (data[0] - (m0 + m1 * data[1] - m2 * data[2])), 50.0))),
    (rates, means, volatilities),
    training_pl[ws.outputs[2]].to_numpy(),
    p0=[1.0, 3.0, 1.0, 0.05],
    maxfev=20000,
)[0]
PARAMS = {
    "return_intercept": round(float(return_intercept), 4),
    "return_slope": round(float(return_slope), 4),
    "variance_square": round(float(variance_square), 4),
    "variance_linear": round(float(variance_linear), 4),
    "variance_constant": round(float(variance_constant), 4),
    "slope": round(float(fitted[0]), 4),
    "m0": round(float(fitted[1]), 4),
    "m1": round(float(fitted[2]), 4),
    "m2": round(float(fitted[3]), 4),
}
print("fitted", PARAMS)
assert PARAMS["return_intercept"] == 2.0 and PARAMS["return_slope"] == 4.0, "the fit did not recover the asset returns"
assert PARAMS["slope"] > 0.0, "the shortfall probability does not rise with the withdrawal rate"

## Evaluate ([6.7.5](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#675-evaluate))

In [ ]:
from workbench import evaluate_check, evaluate_measure, evaluate_predict

MODEL_TEMPLATE = """from math import exp, sqrt

import pandas as pd
from mlflow.models import set_model
from mlflow.pyfunc import PythonModel

INPUT_SHARE = {inputs[0]!r}
INPUT_RATE = {inputs[1]!r}
OUTPUT_RETURN = {outputs[0]!r}
OUTPUT_VOLATILITY = {outputs[1]!r}
OUTPUT_SHORTFALL = {outputs[2]!r}
RETURN_INTERCEPT = {return_intercept!r}
RETURN_SLOPE = {return_slope!r}
VARIANCE_SQUARE = {variance_square!r}
VARIANCE_LINEAR = {variance_linear!r}
VARIANCE_CONSTANT = {variance_constant!r}
SLOPE = {slope!r}
M0 = {m0!r}
M1 = {m1!r}
M2 = {m2!r}


def profile(share, rate):
    weight = share / 100.0
    mean = RETURN_INTERCEPT + RETURN_SLOPE * weight
    volatility = sqrt(max(VARIANCE_SQUARE * weight**2 + VARIANCE_LINEAR * weight + VARIANCE_CONSTANT, 0.0))
    shortfall = 100.0 / (1.0 + exp(min(-SLOPE * (rate - (M0 + M1 * mean - M2 * volatility)), 50.0)))
    return mean, volatility, shortfall


class Profile(PythonModel):
    def predict(self, context, model_input: pd.DataFrame, params=None):
        results = [profile(share, rate) for share, rate in zip(model_input[INPUT_SHARE], model_input[INPUT_RATE], strict=True)]
        return pd.DataFrame(
            {{
                OUTPUT_RETURN: [result[0] for result in results],
                OUTPUT_VOLATILITY: [result[1] for result in results],
                OUTPUT_SHORTFALL: [result[2] for result in results],
            }}
        )


set_model(Profile())
"""

model_path = join(ws.path, "model.py")
with open(model_path, "w") as model_file:
    model_file.write(MODEL_TEMPLATE.format(**PARAMS, inputs=ws.inputs, outputs=ws.outputs))

predict = evaluate_predict(model_path, "pyfunc", ws.signature)

LIMITS = {ws.outputs[0]: 0.05, ws.outputs[1]: 0.05, ws.outputs[2]: 10.0}
SPLITS = (("validation", validation_pl), ("backtest", backtest_pl))


def acceptance_checks():
    answers = {split: predict({name: frame[name].to_list() for name in ws.inputs}) for split, frame in SPLITS}
    return [
        {"split": split, "output": output, "name": "rmse", "value": evaluate_measure(frame[output].to_list(), answers[split][output])["rmse"], "limit": LIMITS[output], "direction": "max"}
        for split, frame in SPLITS
        for output in ws.outputs
    ]


evaluation = evaluate_check(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "passed", entry["passed"])
print("acceptance", "passed" if evaluation["passed"] else "failed, the shipped model is outside the limit")

## Package ([6.7.6](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#676-package))

In [ ]:
from workbench import package_workspace


def golden_samples():
    shares = [0.0, 25.0, 50.0, 75.0, 100.0]
    rates = [2.0, 3.0, 4.0, 5.0, 6.0]
    answers = predict({ws.inputs[0]: shares, ws.inputs[1]: rates})
    return {
        ws.inputs[0]: [*shares, None],
        ws.inputs[1]: [*rates, rates[0]],
        **{output: [*answers[output], None] for output in ws.outputs},
        "expect": ["ok"] * len(shares) + ["error"],
    }


golden = golden_samples()
package_workspace(
    ws,
    "pyfunc",
    golden,
    trained_on="synthetic two asset blend and seeded simulation, training split",
    inputs=[training_path],
    notes="line and quadratic for the mix, logistic curve for the shortfall",
    implementation="mlflow.pyfunc model.py, requires mlflow, pandas and the standard library",
    model_path=model_path,
    evaluation=evaluation,
    sequence="across the range of the inputs",
)

## Register ([6.7.7](https://github.com/ggear/asystem/blob/master/src/rue/mlmodels/src/build/resources/doc/model.md#677-register))

In [ ]:
from workbench import register_workspace

register_workspace(ws)